# 🚀 CIFAR-100 Training on Colab

自動掛載 Google Drive、clone repo、偵測 checkpoint 並恢復訓練。

| Step | 說明 | 執行時機 |
|------|------|----------|
| 1 | 掛載 Google Drive | 每次啟動 |
| 2 | Clone / Pull 程式碼 | 每次啟動 |
| 3 | 安裝依賴套件 | 每次啟動 |
| 4 | 確認 GPU 環境 | 每次啟動 |
| 5 | 自動偵測 Checkpoint 並訓練 | 訓練時 |
| 6 | 確認 Checkpoint 已存到 Drive | 訓練後 |
| 7 | 執行模型評量 | 訓練完成後 |
| 8 | 清除舊 Checkpoint（選用） | 需重頭訓練時 |

## Step 1｜掛載 Google Drive

In [34]:
from google.colab import drive
drive.mount('/content/drive')

import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoint 目錄：{CHECKPOINT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Checkpoint 目錄：/content/drive/MyDrive/cifar100_checkpoints


## Step 2｜Clone / Pull 最新程式碼

In [35]:
import os

!git config --global user.name "rvgoing"
!git config --global user.email "cmkdkimo@gmail.com"

GITHUB_REPO = 'https://github.com/rvgoing/CIFARMLOps.git'
REPO_NAME   = 'CIFARMLOps'

if os.path.exists(f'/content/{REPO_NAME}'):
    print('📦 Repo 已存在，執行 git pull ...')
    %cd /content/{REPO_NAME}
    !git pull
else:
    print('📦 Clone repo ...')
    !git clone {GITHUB_REPO}
    %cd /content/{REPO_NAME}

print('✅ 程式碼已是最新版本')

📦 Repo 已存在，執行 git pull ...
/content/CIFARMLOps
Already up to date.
✅ 程式碼已是最新版本


## Step 3｜安裝依賴套件

In [36]:
import subprocess
import sys

print('📦 開始安裝套件...')

with open('requirements.txt') as f:
    packages = [line.strip() for line in f if line.strip() and not line.startswith('#')]

for pkg in packages:
    print(f'  安裝 {pkg} ...', end=' ')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('✅')
    else:
        print('❌')
        print(result.stderr)

print('\n✅ 所有套件安裝完成')

📦 開始安裝套件...
  安裝 torch>=2.0.0 ... ✅
  安裝 torchvision>=0.15.0 ... ✅
  安裝 tqdm ... ✅
  安裝 matplotlib>=3.0 ... ✅
  安裝 scikit-learn>=1.0 ... ✅
  安裝 mlflow>=3.0 ... ✅
  安裝 gradio>=4.0 ... ✅
  安裝 Pillow>=9.0 ... ✅

✅ 所有套件安裝完成


## Step 4｜確認 GPU 環境

In [37]:
import torch

print(f'PyTorch 版本：{torch.__version__}')
print(f'CUDA 可用：{torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU：{torch.cuda.get_device_name(0)}')
    print(f'VRAM：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  未偵測到 GPU，請確認 Runtime > Change runtime type > T4 GPU')

!nvidia-smi

PyTorch 版本：2.10.0+cu128
CUDA 可用：True
GPU：Tesla T4
VRAM：14.6 GB
Wed May  6 04:11:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |         

## Step 5｜自動偵測 Checkpoint 並開始訓練

> 自動判斷是否有 `checkpoint.pth` 或 `model_best.pth`，有則恢復訓練，無則從頭開始。

In [38]:
import os

# 確保變數在此 cell 可用（若從 Step 1 重新執行需重新定義）
CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
BEST_CKPT  = os.path.join(CHECKPOINT_DIR, 'model_best.pth')
LAST_CKPT  = os.path.join(CHECKPOINT_DIR, 'checkpoint.pth')
MLFLOW_DIR = os.path.join(CHECKPOINT_DIR, 'mlruns')

# 優先使用 last checkpoint（含 optimizer state），其次 best checkpoint
if os.path.exists(LAST_CKPT):
    resume_path = LAST_CKPT
    print(f'🔄 發現 checkpoint，將從上次進度恢復：{resume_path}')
elif os.path.exists(BEST_CKPT):
    resume_path = BEST_CKPT
    print(f'🔄 發現 best checkpoint，將從此恢復：{resume_path}')
else:
    resume_path = ''
    print('🆕 未發現 checkpoint，從頭開始訓練')

resume_arg = f'--resume {resume_path}' if resume_path else ''

cmd = (
    f"python train.py "
    f"--epochs 10 "
    f"--batch-size 128 "
    f"--lr 0.1 "
    f"--num-workers 2 "
    f"--save-dir {CHECKPOINT_DIR} "
    f"--mlflow-dir {MLFLOW_DIR} "
    f"{resume_arg}"
)

print(f'\n🚀 執行指令：\n{cmd}\n')
!{cmd}

🆕 未發現 checkpoint，從頭開始訓練

🚀 執行指令：
python train.py --epochs 10 --batch-size 128 --lr 0.1 --num-workers 2 --save-dir /content/drive/MyDrive/cifar100_checkpoints --mlflow-dir /content/drive/MyDrive/cifar100_checkpoints/mlruns 

Using device: cuda
MLflow Local Dir: /content/drive/MyDrive/cifar100_checkpoints/mlruns
/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Epoch [1/10]
Train Loss: 3.9720 | Train Acc: 8.87%
Val   Loss: 3.5734 | Val   Acc: 13.43%
Epoch [2/10]
Train Loss: 3.3252 | Train Acc: 18.78%
Val   Loss: 3.1705 | Val   Acc: 21.68%
Epoch [3/10]
Train Loss: 2.8159 | Train Acc: 28.29%
Val  

## Step 6｜確認 Checkpoint 已存到 Drive

In [39]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
MLFLOW_DIR = os.path.join(CHECKPOINT_DIR, 'mlruns')

print('📂 Drive checkpoint 目錄內容：')
files = sorted(os.listdir(CHECKPOINT_DIR))

if not files:
    print('  （目錄為空）')
else:
    for f in files:
        fpath = os.path.join(CHECKPOINT_DIR, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1024**2
            print(f'  {f:<30} ({size:.1f} MB)')
        else:
            print(f'  {f}/')

print('\n📊 MLflow 訓練記錄 (mlruns) 目錄內容：')
if os.path.exists(MLFLOW_DIR) and os.listdir(MLFLOW_DIR):
    for root, dirs, files in os.walk(MLFLOW_DIR):
        level = root.replace(MLFLOW_DIR, '').count(os.sep)
        indent = '  ' * (level + 1)
        print(f'{indent}{os.path.basename(root)}/')
        for f in files:
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath) / 1024**2
            print(f'{indent}  {f:<28} ({size:.1f} MB)')
else:
    print('  （MLflow 記錄目錄為空或不存在）')

📂 Drive checkpoint 目錄內容：
  checkpoint.pth                 (85.7 MB)
  eval_results/
  mlruns/
  model_best.pth                 (85.7 MB)
  training_log.json              (0.0 MB)

📊 MLflow 訓練記錄 (mlruns) 目錄內容：
  mlruns/
    0/
      meta.yaml                    (0.0 MB)
    .trash/
    564914154859694942/
      meta.yaml                    (0.0 MB)
      38d9c2af7b6e4b6eb3d2dcc00d603331/
        meta.yaml                    (0.0 MB)
        metrics/
          train_loss                   (0.0 MB)
          train_acc                    (0.0 MB)
          val_acc                      (0.0 MB)
          val_loss                     (0.0 MB)
        params/
          epochs                       (0.0 MB)
          batch_size                   (0.0 MB)
          lr                           (0.0 MB)
          momentum                     (0.0 MB)
          weight_decay                 (0.0 MB)
          optimizer                    (0.0 MB)
          scheduler                    (0.0 MB)
   

## Step 7｜執行評量（訓練完成後執行）

In [40]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
BEST_CKPT  = os.path.join(CHECKPOINT_DIR, 'model_best.pth')
LAST_CKPT  = os.path.join(CHECKPOINT_DIR, 'checkpoint.pth')
EVAL_DIR   = os.path.join(CHECKPOINT_DIR, 'eval_results')

os.makedirs(EVAL_DIR, exist_ok=True)

# Determine which checkpoint to use for evaluation
eval_checkpoint_path = ""
if os.path.exists(BEST_CKPT):
    eval_checkpoint_path = BEST_CKPT
    print(f'🔍 發現最佳模型 checkpoint，將以此進行評量：{eval_checkpoint_path}')
elif os.path.exists(LAST_CKPT):
    eval_checkpoint_path = LAST_CKPT
    print(f'🔍 發現上次訓練 checkpoint，將以此進行評量：{eval_checkpoint_path}')
else:
    print('⚠️  找不到任何有效的 checkpoint (model_best.pth 或 checkpoint.pth)，請先完成訓練（Step 5）')

if eval_checkpoint_path:
    cmd = f"python evaluate.py --checkpoint {eval_checkpoint_path} --save-dir {EVAL_DIR}"
    print(f'\n🔍 執行評量指令：\n{cmd}\n')
    !{cmd}

    print('\n📂 評量結果：')
    files_in_eval_dir = sorted(os.listdir(EVAL_DIR))
    if not files_in_eval_dir:
        print('  （評量目錄為空）')
    else:
        for f in files_in_eval_dir:
            fpath = os.path.join(EVAL_DIR, f)
            if os.path.isfile(fpath):
                size = os.path.getsize(fpath) / 1024**2
                print(f'  {f:<30} ({size:.1f} MB)')
            else:
                print(f'  {f}/')

🔍 發現最佳模型 checkpoint，將以此進行評量：/content/drive/MyDrive/cifar100_checkpoints/model_best.pth

🔍 執行評量指令：
python evaluate.py --checkpoint /content/drive/MyDrive/cifar100_checkpoints/model_best.pth --save-dir /content/drive/MyDrive/cifar100_checkpoints/eval_results


========== CIFAR-100 Evaluation ==========
[1/5] Loading model ...
Loading checkpoint: /content/drive/MyDrive/cifar100_checkpoints/model_best.pth
  Loaded from epoch 9, best_acc=50.50%
[2/5] Running inference ...
[3/5] Computing metrics ...
      Top-1 Accuracy : 50.50%
      Top-5 Accuracy : 79.73%
[4/5] Saving plots ...
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/loss_acc_curve.png
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/confusion_matrix.png
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/per_class_accuracy.png
[5/5] Saving reports ...
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/classification_report.txt
  Saved: /content/drive/MyDrive/cifar10

## Step 8｜清除舊的 Checkpoint（需要時才執行）

> ⚠️ **警告**：執行此 cell 將刪除訓練進度，下次訓練將從頭開始。

In [41]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'

# 安全確認
CONFIRM = False  # 改為 False 可防止意外執行

if not CONFIRM:
    print('⛔ 未確認，跳過刪除。請將 CONFIRM 改為 True 後再執行。')
else:
    targets = ['checkpoint.pth', 'model_best.pth', 'training_log.json']
    for f in targets:
        path = os.path.join(CHECKPOINT_DIR, f)
        if os.path.exists(path):
            os.remove(path)
            print(f'🗑️  已刪除：{f}')
        else:
            print(f'⏭️  不存在，跳過：{f}')
    print('\n✅ 完成，可以從頭訓練了')

🗑️  已刪除：checkpoint.pth
🗑️  已刪除：model_best.pth
🗑️  已刪除：training_log.json

✅ 完成，可以從頭訓練了
